In [1]:
# Load Packages
library(Pando)
library(Seurat)
library(Matrix)
library(data.table)
library(BSgenome.Hsapiens.UCSC.hg38)
library(EnsDb.Hsapiens.v86)
library(Signac)
library(TFBSTools) # 处理 motifs 对象需要
library(GenomeInfoDb) # <--- 新增：必须加载，否则 keepStandardChromosomes 会报错
library(dplyr)
data(motifs)


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:Pando’:

    LayerData, VariableFeatures


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘Seurat’


The following objects are masked from ‘package:Pando’:

    GetAssay, VariableFeatures


Loading required package: GenomeInfoDb

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Po

In [3]:
cell_type <- "HepG2"

data_path <-paste0("/home/liyang/BioWuYan/dygmamba_project/data/cell_line/", cell_type, "/process/")

output_path <- paste0("/home/liyang/BioWuYan/dygmamba_project/data/cell_line/", cell_type, "/data_pando/")


grn_obj <- readRDS(paste0(output_path,"grn_seurat_object_tt.rds"))

In [22]:
# 1. 提取所有推断出的调控参数
grn_df <- coef(grn_obj)

# 2. 转换为标准的 DataFrame
grn_df <- as.data.frame(grn_df)

grn_significant <- subset(grn_df, padj < 0.05)

tf_gene_df <- grn_significant[, c('tf', 'target', 'estimate', 'padj')]
colnames(tf_gene_df) <- c('TF', 'Gene', 'Weight', 'padj')
tf_gene_df <- tf_gene_df %>% 
  distinct(TF, Gene, .keep_all = TRUE)

tf_region_df <- grn_significant[, c('tf', 'region', 'estimate', 'padj')]
colnames(tf_region_df) <- c('TF', 'Region', 'Weight', 'padj')
tf_region_df <- tf_region_df %>% 
  distinct(TF, Region, .keep_all = TRUE)

region_gene_df <- grn_significant[, c('region', 'target', 'estimate', 'padj')]
colnames(region_gene_df) <- c('Region', 'Gene', 'Weight', 'padj')
region_gene_df <- region_gene_df %>% 
  distinct(Region, Gene, .keep_all = TRUE)

write.csv(tf_gene_df, file = paste0(output_path, "tf_gene_network.csv"), row.names = FALSE)
write.csv(tf_region_df, file = paste0(output_path, "tf_region_network.csv"), row.names = FALSE)
write.csv(region_gene_df, file = paste0(output_path, "region_gene_network.csv"), row.names = FALSE)

In [ ]:

# 依据 term (TF) 和 region (Peak) 删除重复



原始行数: 105

去重后行数: 105



In [16]:
tf_gene_df
tf_region_df
region_gene_df

,TF,Gene,Weight,padj
,<chr>,<chr>,<dbl>,<dbl>
109,ZNF292,CDC25B,0.22729104,0.012731249
196,MZF1,SRC,0.12554847,0.021478433
288,FOXP4,TAF4,-0.21165794,0.024474163
462,ATF7,RERE,-0.57530693,0.009165318
541,ZFP64,FUCA1,0.30105550,0.019712886
554,RARA,DHDDS,-0.10380162,0.021859901
601,ZNF689,MARCKSL1,-0.27367482,0.014161315
612,ZNF765,PHC2,-0.26775717,0.010730838
675,REPIN1,BTBD19,0.10130549,0.040474106


,TF,Region,Weight,padj
,<chr>,<chr>,<dbl>,<dbl>
109,ZNF292,chr20-3739193-3740578,0.22729104,0.012731249
196,MZF1,chr20-37332031-37332630,0.12554847,0.021478433
288,FOXP4,chr20-62104409-62104736,-0.21165794,0.024474163
462,ATF7,chr1-8626029-8627024,-0.57530693,0.009165318
541,ZFP64,chr1-23765494-23766246,0.30105550,0.019712886
554,RARA,chr1-26350735-26352137,-0.10380162,0.021859901
601,ZNF689,chr1-32338424-32338916,-0.27367482,0.014161315
612,ZNF765,chr1-33347083-33347860,-0.26775717,0.010730838
675,REPIN1,chr1-44793273-44794296,0.10130549,0.040474106


,Region,Gene,Weight,padj
,<chr>,<chr>,<dbl>,<dbl>
109,chr20-3739193-3740578,CDC25B,0.22729104,0.012731249
196,chr20-37332031-37332630,SRC,0.12554847,0.021478433
288,chr20-62104409-62104736,TAF4,-0.21165794,0.024474163
462,chr1-8626029-8627024,RERE,-0.57530693,0.009165318
541,chr1-23765494-23766246,FUCA1,0.30105550,0.019712886
554,chr1-26350735-26352137,DHDDS,-0.10380162,0.021859901
601,chr1-32338424-32338916,MARCKSL1,-0.27367482,0.014161315
612,chr1-33347083-33347860,PHC2,-0.26775717,0.010730838
675,chr1-44793273-44794296,BTBD19,0.10130549,0.040474106
